# Dreamer: Batch User Memory Distillation Pipeline

Nightly job that:
1. Goes through all child branches of the `production` Lakebase branch (new agent sessions)
2. For each child branch/agent session, fetches the last 24 hours of chat logs
3. Distills chat conversation into episodic, semantic, and procedural memories 
4. Updates the user memories in the **production** branch (snapshot of branch is taken before memories are written)
5. Logs all operations to a file for observability

Note: In your Job Task, make sure to configure your environment such that it includes the following dependencies:
1. databricks-langchain[memory]>=0.17.0
2. databricks-sdk>=0.94.0

In [ ]:
import json
import os
import asyncio
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor

from databricks.sdk import WorkspaceClient
from databricks_langchain import ChatDatabricks, AsyncDatabricksStore
from databricks_ai_bridge.lakebase import LakebaseClient
from langchain_core.messages import SystemMessage, HumanMessage

## 1. Configuration

In [ ]:
# -- Lakebase project & branch config --
LAKEBASE_PROJECT = "memories-user" # TODO: Update lakebase project name here
PRODUCTION_BRANCH = "production"

# -- LLM config --
llm = ChatDatabricks(endpoint="databricks-claude-opus-4-7", max_tokens=2000)

# -- Embedding config for the memory store --
EMBEDDING_ENDPOINT = "databricks-qwen3-embedding-0-6b"
EMBEDDING_DIMS = 1024

# -- Schema where the vector extension and store tables live --
MEMORY_SCHEMA = "memories"

# -- Observability log file (persisted to UC Volume) --
_log_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
LOG_DIR = "/Volumes/catalog/schema/dreamer_logs" # TODO: Update volume directory here
LOG_FILE = f"{LOG_DIR}/dreamer_user_distillation_{_log_ts}.md"

# -- Databricks SDK client --
w = WorkspaceClient()

## 2. Observability Logger

In [ ]:
distillation_log = []


def log_event(event_type: str, branch: str, user: str = "", detail: str = ""):
    """Append a structured log entry to the in-memory log."""
    ts = datetime.now().isoformat()
    entry = f"[{ts}] [{event_type}] branch={branch} user={user} | {detail}"
    distillation_log.append(entry)
    print(entry)

## 3. Discover Child Branches of Production

In [ ]:
def list_child_branches(project: str, parent_branch: str) -> list:
    """
    List all branches in the project whose source (parent) is the given parent_branch.
    """
    parent_resource = f"projects/{project}"
    production_resource = f"projects/{project}/branches/{parent_branch}"

    all_branches = list(w.postgres.list_branches(parent=parent_resource))

    child_branches = []
    for branch in all_branches:
        if branch.name == production_resource:
            continue
        # source_branch is nested under branch.status
        status = branch.status if hasattr(branch, "status") else None
        if status is None:
            continue
        source = getattr(status, "source_branch", None) if not isinstance(status, dict) else status.get("source_branch")
        if source == production_resource:
            child_branches.append(branch)

    return child_branches


def branch_id_from_name(branch_name: str) -> str:
    """Extract the branch_id from a full resource name like 'projects/foo/branches/bar'."""
    return branch_name.rsplit("/", 1)[-1]

## 4. Memory Tools (Save & Delete)

In [ ]:
async def save_user_memory(
    memory_key: str, memory_data: dict, store, user_id: str, branch_name: str
) -> str:
    namespace = ("user_memories", user_id)
    try:
        await store.aput(namespace, memory_key, memory_data)
        msg = f"SAVED memory '{memory_key}' for {user_id}: {json.dumps(memory_data)}"
        log_event("SAVE_MEMORY", branch_name, user_id, msg)
        return msg
    except Exception as e:
        msg = f"FAILED to save memory '{memory_key}' for {user_id}: {e}"
        log_event("SAVE_MEMORY_ERROR", branch_name, user_id, msg)
        return msg


async def delete_user_memory(
    memory_key: str, store, user_id: str, branch_name: str
) -> str:
    namespace = ("user_memories", user_id)
    try:
        await store.adelete(namespace, memory_key)
        msg = f"DELETED memory '{memory_key}' for {user_id}"
        log_event("DELETE_MEMORY", branch_name, user_id, msg)
        return msg
    except Exception as e:
        msg = f"FAILED to delete memory '{memory_key}' for {user_id}: {e}"
        log_event("DELETE_MEMORY_ERROR", branch_name, user_id, msg)
        return msg

## 5. Extract 24-Hour Logs from a Branch

In [ ]:
def _fetch_daily_chat_logs_sync(project: str, branch: str) -> dict:
    """Query the last 24 hours of chat messages from a specific Lakebase branch (sync)."""
    with LakebaseClient(project=project, branch=branch) as client:
        query = """
            SELECT c."userId", m."chatId", m.role, m.parts
            FROM ai_chatbot."Message" m
            JOIN ai_chatbot."Chat" c ON m."chatId" = c.id
            WHERE m."createdAt" >= NOW() - INTERVAL '1 day'
            ORDER BY c."userId", m."chatId", m."createdAt";
        """
        rows = client.execute(query) or []

    # Group logs by userId (e.g. "4255605719359984@2850744067564480")
    user_logs = {}
    for row in rows:
        user_id = str(row["userId"])
        if user_id not in user_logs:
            user_logs[user_id] = []
        user_logs[user_id].append(f"{row['role'].upper()}: {json.dumps(row['parts'])}")

    return user_logs


async def fetch_daily_chat_logs(project: str, branch: str) -> dict:
    """Run the sync LakebaseClient query in a thread to avoid blocking the async event loop."""
    loop = asyncio.get_event_loop()
    return await loop.run_in_executor(None, _fetch_daily_chat_logs_sync, project, branch)

## 6. LLM Distillation & Execution

In [ ]:
SYSTEM_PROMPT = """
You are an AI Memory Distillation Engine. Review the following 24-hour chat history for a user.
Identify explicit user preferences, facts, or commands to forget information.

Categorize saved memories strictly as:
- "semantic" (immutable facts, e.g., role, languages)
- "episodic" (recent events, projects)
- "procedural" (how they prefer the agent to behave/format)

Respond ONLY with a valid JSON array of tool calls:
[
  {
    "tool": "save_user_memory",
    "memory_key": "favorite_language",
    "memory_data": {"value": "Python", "category": "semantic"}
  },
  {
    "tool": "delete_user_memory",
    "memory_key": "old_project"
  }
]

If there are no memories to save or delete, respond with an empty array: []
"""


async def distill_branch(branch_name: str, user_logs: dict, store):
    """Run distillation for all users on a single branch."""
    for user_id, transcripts in user_logs.items():
        log_event("PROCESSING_USER", branch_name, user_id, f"{len(transcripts)} messages")
        chat_text = "\n".join(transcripts)

        messages = [
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=f"Chat History:\n{chat_text}"),
        ]

        try:
            response = llm.invoke(messages)
            raw_output = response.content.strip().replace("```json", "").replace("```", "")
            actions = json.loads(raw_output)
            log_event("LLM_RESPONSE", branch_name, user_id, f"{len(actions)} actions extracted")

            for action in actions:
                if action["tool"] == "save_user_memory":
                    await save_user_memory(
                        memory_key=action["memory_key"],
                        memory_data=action["memory_data"],
                        store=store,
                        user_id=user_id,
                        branch_name=branch_name,
                    )
                elif action["tool"] == "delete_user_memory":
                    await delete_user_memory(
                        memory_key=action["memory_key"],
                        store=store,
                        user_id=user_id,
                        branch_name=branch_name,
                    )

        except Exception as e:
            log_event("USER_ERROR", branch_name, user_id, str(e))

## 7. Main Pipeline: Loop Over Child Branches

In [ ]:
async def run_distillation_pipeline():
    log_event("PIPELINE_START", PRODUCTION_BRANCH, detail="Discovering child branches...")

    child_branches = list_child_branches(LAKEBASE_PROJECT, PRODUCTION_BRANCH)
    branch_names = [branch_id_from_name(b.name) for b in child_branches]
    log_event(
        "BRANCHES_FOUND",
        PRODUCTION_BRANCH,
        detail=f"{len(child_branches)} child branches: {branch_names}",
    )

    if not child_branches:
        log_event("PIPELINE_END", PRODUCTION_BRANCH, detail="No child branches found. Nothing to do.")
        return

    # Open the production store once — all memories get written here
    # schema=MEMORY_SCHEMA ensures the vector extension is on the search path
    async with AsyncDatabricksStore(
        project=LAKEBASE_PROJECT,
        branch=PRODUCTION_BRANCH,
        embedding_endpoint=EMBEDDING_ENDPOINT,
        embedding_dims=EMBEDDING_DIMS,
        schema=MEMORY_SCHEMA,
    ) as store:
        await store.setup()

        for branch in child_branches:
            branch_name = branch_id_from_name(branch.name)
            log_event("BRANCH_START", branch_name, detail="Fetching chat logs...")

            try:
                user_logs = await fetch_daily_chat_logs(LAKEBASE_PROJECT, branch_name)
                log_event(
                    "BRANCH_LOGS",
                    branch_name,
                    detail=f"{len(user_logs)} users with messages",
                )

                if user_logs:
                    await distill_branch(branch_name, user_logs, store)

                log_event("BRANCH_DONE", branch_name, detail="Complete.")

            except Exception as e:
                log_event("BRANCH_ERROR", branch_name, detail=str(e))

    log_event("PIPELINE_END", PRODUCTION_BRANCH, detail="All branches processed.")

## 8. Run the Pipeline

In [ ]:
await run_distillation_pipeline()

## 9. Distillation Summary

In [ ]:
# Build clickable URL to view the file in the Databricks workspace UI
def _make_volume_url(file_path: str) -> str:
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    workspace_id = ctx.workspaceId().get()
    host = w.config.host.rstrip("/")
    # file_path is like /Volumes/<catalog>/<schema>/<volume>/<filename>
    parts = file_path.removeprefix("/Volumes/").split("/", 3)
    volume_dir = "/".join(parts[:3])
    file_name = parts[3] if len(parts) > 3 else ""
    return f"{host}/explore/data/volumes/{volume_dir}?o={workspace_id}&filePreviewPath={file_name}"


# Build a markdown summary and append it to the log file
saves = [e for e in distillation_log if "SAVE_MEMORY]" in e]
deletes = [e for e in distillation_log if "DELETE_MEMORY]" in e]
errors = [e for e in distillation_log if "ERROR" in e]

summary_lines = [
    f"# Dreamer Distillation Summary",
    f"**Run timestamp:** {_log_ts}",
    f"**Log file:** `{LOG_FILE}`",
    "",
    f"| Metric | Count |",
    f"|--------|-------|",
    f"| Total log entries | {len(distillation_log)} |",
    f"| Memories saved | {len(saves)} |",
    f"| Memories deleted | {len(deletes)} |",
    f"| Errors | {len(errors)} |",
    "",
]

if saves:
    summary_lines += ["## Saves", "```"] + saves + ["```", ""]
if deletes:
    summary_lines += ["## Deletes", "```"] + deletes + ["```", ""]
if errors:
    summary_lines += ["## Errors", "```"] + errors + ["```", ""]

summary_lines += ["## Full Log", "```"] + distillation_log + ["```"]

summary_md = "\n".join(summary_lines)

# Write the final markdown summary to the UC Volume
with open(LOG_FILE, "w") as f:
    f.write(summary_md)

print(summary_md)
print(f"\nLog written to: {LOG_FILE}")
print(f"View log at: {_make_volume_url(LOG_FILE)}")